# MILP Optimization: Solving the TSP in Cabárceno

In this notebook, we tackle the mathematical core of the problem. We will model the visit to the park as a **Traveling Salesperson Problem (TSP)** on an asymmetrical graph.

Our goal is to find the decision matrix $x_{ij} \in \{0,1\}$ that minimizes the total distance traveled:

$$\min \sum_{i=1}^{n} \sum_{j=1}^{n} c_{ij} x_{ij}$$

Where $c_{ij}$ is the actual distance in meters (calculated via Dijkstra on the OpenStreetMap graph) between enclosure $i$ and enclosure $j$. We will use **Google OR-Tools** to solve the system using Mixed Integer Linear Programming (MILP).

In [20]:
import osmnx as ox
import networkx as nx
import numpy as np
import pandas as pd
import folium
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import geopandas as gpd
from shapely.ops import nearest_points
from pyproj import Transformer
from itertools import product as iproduct
import webbrowser, os
from collections import defaultdict

In [21]:
# Load the projected graph in meters (UTM) saved previously
G_metros = ox.load_graphml('../data/processed/cabarceno_graph.graphml')

print(f"Graph loaded. Nodes: {len(G_metros.nodes)}, Edges: {len(G_metros.edges)}")
print(f"CRS: {G_metros.graph['crs']}")

Graph loaded. Nodes: 1930, Edges: 2679
CRS: EPSG:32630


## 1. Mapping Enclosures to Graph Nodes

We define the GPS coordinates (Longitude, Latitude) of the animals we want to visit today and find the closest mathematical node in our road network.

In [22]:
# Search for park enclosures using the tags that appear in OSM:
# meadows, scrub, buildings, and tourist attractions
tags_enclosures = {
    'landuse': ['meadow', 'grass'],
    'natural': 'scrub',
    'building': True,
    'tourism': 'attraction'
}

areas_gdf = ox.features_from_place("Parque de la Naturaleza de Cabárceno, Cantabria, Spain", tags_enclosures)

# Keep only those with a name
named_enclosures = areas_gdf[areas_gdf['name'].notna()].copy()

# If there are duplicate names, add a numeric suffix to distinguish them (e.g., "Llama (1)", "Llama (2)")
counts = {}
unique_names = []
for name in named_enclosures['name']:
    counts[name] = counts.get(name, 0) + 1
    unique_names.append(name)

seen = {}
final_names = []
for name in unique_names:
    if counts[name] > 1:
        seen[name] = seen.get(name, 0) + 1
        final_names.append(f"{name} ({seen[name]})")
    else:
        final_names.append(name)

named_enclosures = named_enclosures.copy()
named_enclosures['name'] = final_names

# CLUSTERING LOGIC
to_drop = [
    "Casa de cebras Grevy",
    "Exhibiciones Aves rapaces",
    "Aula de Educación Medioambiental",
    "Granja",
    "Centro de Recuperación de Fauna Silvestre",
    "Leones Marinos",
    "Tigre",
    "Gorilas",
    "Llama (2)",
    "Jaguar"
]
named_enclosures = named_enclosures[~named_enclosures['name'].isin(to_drop)].copy()

renames = {
    "Aves rapaces": "Aves rapaces/Granja",
    "Reptilario": "Reptilario/Leones Marinos",
    "Gorila": "Gorila/Tigre",
    "Llama (1)": "Llama",
    "Watusi (1)": "Watusi",
    "Watusi (2)": "Dromedario"
}
named_enclosures['name'] = named_enclosures['name'].replace(renames)

print(f"Named enclosures found: {len(named_enclosures)}")
print(named_enclosures['name'].tolist())

Named enclosures found: 30
['Vaca y Caballo Monchino', 'Llama', 'Camello Bactriano', 'Fauna Ibérica', 'Gaur', 'Gorila/Tigre', 'Rinoceronte Blanco', 'Vaca Tudanca', 'Asno Somalí', 'Cebra', 'Hipopótamo', 'Jirafa, Avestruz y Eland', 'Addax y Búfalo Cafre', 'Bisonte Europeo', 'Cebras Grevy', 'Elefante Africano, Búfalo de agua y Cobo de Lechwe', 'Oryx del Cabo', 'Watusi', 'Hipopótamos pigmeos', 'Oso', 'Cobos de agua', 'León', 'Lobo', 'Papion de Guinea', 'Wallaby de Bennet y Emú', 'Dromedario', 'Aves rapaces/Granja', 'Guepardo', 'Linces', 'Reptilario/Leones Marinos']


In [23]:
# Download all parking lots inside the park
parkings_gdf = ox.features_from_place(
    "Parque de la Naturaleza de Cabárceno, Cantabria, Spain",
    tags={'amenity': 'parking'}
)

# Calculate the centroid of each parking lot in UTM
parkings_proj = parkings_gdf.to_crs(G_metros.graph['crs'])
parkings_centroids_utm = parkings_proj.copy()
parkings_centroids_utm.geometry = parkings_proj.geometry.centroid

In [24]:
# Main Entrance: OSM parking way 358294677
entrance_centroid_utm = parkings_centroids_utm.loc[('way', 358294677), 'geometry']

enclosures_proj = named_enclosures.to_crs(G_metros.graph['crs'])

# To facilitate joins and extract geometries, we flatten the DataFrames' indices
parkings_flat = parkings_proj.reset_index()
enclosures_flat = enclosures_proj.reset_index(drop=True)

# 1. Get the closest parking regardless of distance
closest_parking = gpd.sjoin_nearest(enclosures_flat, parkings_flat, max_distance=None, distance_col="dist")

# 2. Get ALL parking lots within a maximum of M=50 meters
# We use a 50m buffer to find all intersecting parkings, as sjoin_nearest only finds the single nearest
M = 50
enclosures_buffered = enclosures_flat.copy()
enclosures_buffered.geometry = enclosures_buffered.geometry.buffer(M)
close_M = gpd.sjoin(enclosures_buffered, parkings_flat, how="inner", predicate="intersects")

### 1.1. Extracting Geometric Candidates

With the intersection data calculated, we now iterate over every enclosure name. For any nearby parking lot (within 50m), we find the single closest geometrical point from the enclosure border to that parking polygon, storing the exact `(x_utm, y_utm)` GPS candidate matches.

In [25]:
# Join the found indices for each enclosure
# candidates_utm[name] = list of (x_utm, y_utm) coordinates of the exact points on the parking border
candidates_utm = {"Main Entrance": [(entrance_centroid_utm.x, entrance_centroid_utm.y)]}

for idx, enclosure_row in enclosures_flat.iterrows():
    name = enclosure_row['name']
    geom_enclosure = enclosure_row.geometry
    
    # Collect all unique candidate parking indices for this enclosure
    p_idxs = set(closest_parking.loc[closest_parking.index == idx, 'index_right'])
    
    if idx in close_M.index:
        # Get all parkings that intersect the 50m buffer
        subset_M = close_M.loc[close_M.index == idx]
        if isinstance(subset_M, gpd.GeoDataFrame) or isinstance(subset_M, pd.DataFrame):
            p_idxs.update(subset_M['index_right'])
        else:
            p_idxs.add(subset_M['index_right'])
    
    pts = []
    # Convert each candidate parking lot into an exact point on the enclosure border
    for p_idx in p_idxs:
        geom_parking = parkings_flat.loc[p_idx].geometry
        _, point = nearest_points(geom_enclosure, geom_parking)
        pts.append((point.x, point.y))
        
    candidates_utm[name] = pts

names = list(candidates_utm.keys())

In [26]:
# All unique UTM points (insertion order, no duplicates) → graph nodes
seen_set = set()
all_pts = []
for pts in candidates_utm.values():
    for pt in pts:
        if pt not in seen_set:
            seen_set.add(pt)
            all_pts.append(pt)

x_all = [pt[0] for pt in all_pts]
y_all = [pt[1] for pt in all_pts]
all_nodes = ox.distance.nearest_nodes(G_metros, X=x_all, Y=y_all)
pt_to_node = {pt: node for pt, node in zip(all_pts, all_nodes)}

In [27]:
# candidate_nodes[name] = list of graph nodes, deduplicated, preserving order
candidate_nodes = {}
for name, pts in candidates_utm.items():
    seen_n = set()
    uniq_nodes = []
    for pt in pts:
        nd = pt_to_node[pt]
        if nd not in seen_n:
            seen_n.add(nd)
            uniq_nodes.append(nd)
    candidate_nodes[name] = uniq_nodes

In [28]:
# Pre-compute Dijkstra between all unique candidate nodes
unique_nodes = list(set(nd for ns in candidate_nodes.values() for nd in ns))
dist_between_nodes = {}
paths_between_nodes = {}
for src in unique_nodes:
    lengths, paths = nx.single_source_dijkstra(G_metros, src, weight='length')
    for dst in unique_nodes:
        if dst in lengths:
            dist_between_nodes[(src, dst)] = int(lengths[dst])
            paths_between_nodes[(src, dst)] = paths[dst]

In [29]:
# UTM → WGS84 Transformer for map markers
transformer = Transformer.from_crs(G_metros.graph['crs'], "EPSG:4326", always_xy=True)
pt_to_lonlat = {pt: transformer.transform(pt[0], pt[1]) for pt in all_pts}

total_combos = 1
for name in names:
    total_combos *= len(candidate_nodes[name])

print(f"Unique candidate road nodes: {len(unique_nodes)}")
for name in names:
    n_c = len(candidate_nodes[name])
    print(f"  {'📍' if name == 'Main Entrance' else '🦁'} {name}: {n_c} candidate parking(s)")
print(f"\nPre-computed distances. Total combinations to evaluate: {total_combos}")

Unique candidate road nodes: 52
  📍 Main Entrance: 1 candidate parking(s)
  🦁 Vaca y Caballo Monchino: 3 candidate parking(s)
  🦁 Llama: 1 candidate parking(s)
  🦁 Camello Bactriano: 2 candidate parking(s)
  🦁 Fauna Ibérica: 1 candidate parking(s)
  🦁 Gaur: 1 candidate parking(s)
  🦁 Gorila/Tigre: 2 candidate parking(s)
  🦁 Rinoceronte Blanco: 1 candidate parking(s)
  🦁 Vaca Tudanca: 2 candidate parking(s)
  🦁 Asno Somalí: 1 candidate parking(s)
  🦁 Cebra: 1 candidate parking(s)
  🦁 Hipopótamo: 1 candidate parking(s)
  🦁 Jirafa, Avestruz y Eland: 4 candidate parking(s)
  🦁 Addax y Búfalo Cafre: 2 candidate parking(s)
  🦁 Bisonte Europeo: 1 candidate parking(s)
  🦁 Cebras Grevy: 2 candidate parking(s)
  🦁 Elefante Africano, Búfalo de agua y Cobo de Lechwe: 4 candidate parking(s)
  🦁 Oryx del Cabo: 1 candidate parking(s)
  🦁 Watusi: 2 candidate parking(s)
  🦁 Hipopótamos pigmeos: 1 candidate parking(s)
  🦁 Oso: 1 candidate parking(s)
  🦁 Cobos de agua: 1 candidate parking(s)
  🦁 León: 2 

## 2. Problem Reduction: Exact Distance Matrix

The *solver* doesn't need to know all the curves of Cabárceno, it only requires a **complete graph** with the actual costs $c_{ij}$ between our points of interest. We calculate these costs in meters by applying **Dijkstra's** algorithm.

In [30]:
# Show the base distance matrix: using the closest parking lot to each enclosure (candidate 0)
base_combo = [candidate_nodes[n][0] for n in names]
n_base = len(base_combo)
base_matrix = np.zeros((n_base, n_base), dtype=int)
for i, ni in enumerate(base_combo):
    for j, nj in enumerate(base_combo):
        if i != j:
            base_matrix[i][j] = dist_between_nodes.get((ni, nj), -1)

print("Base distance matrix (in meters), closest parking to each enclosure:")
print(base_matrix)

Base distance matrix (in meters), closest parking to each enclosure:
[[   0 3070 3103 1498 2737 1112 1462 1669 2845 3131 1672 1567 2001 3894
  3379 3860 3546 3060 4658 2502 2029 3248 3398 3060 3647 3014 4207 2272
  2766 4069  155]
 [ 718    0   32 2010 3250 1625 1974 2182 3358 3644 2184 2079 2513 4407
  3891 4372 4058 3573 5171 3015 2541 3760 3911 3573 4160 3526 4720 2784
  3279 4582  743]
 [ 750   32    0 2043 3282 1657 2006 2214 3390 3676 2217 2112 2546 4439
  3923 4405 4091 3605 5203 3047 2574 3793 3943 3605 4192 3559 4752 2817
  3311 4614  776]
 [1564 1572 1604    0 1239  451 1105  171 1347 1633 1010   68 1074 3016
  2500 2981 2619 2176 3780 1618 1198 2369 2520 2176 2769 2135 3329 1345
  1882 3191 1589]
 [1026 3322 3354 1749    0 1528  990 1921  108 3383 2088 1818 2417 4310
  3795 4276 3962 3477 5075 2919 2445 3664 3814 3477 4063 3430 4624 2688
  3183 4485 1051]
 [1112 1957 1990  385 1625    0  654  557 1733 2018  559  454  888 2782
  2266 2747 2433 1948 3546 1390  916 2135 2286 19

## 3. TSP Resolution with Google OR-Tools

We instantiate the routing model. We configure the cost function and apply the Guided Local Search metaheuristic to escape from local minimums and find the optimal visiting order.

In [31]:
# --- Resolution of the Generalized TSP with Disjunctions ---
# Flatten all candidate lists into a single node array (global matrix)
flat_nodes = []
enclosure_to_indices_map = []

global_idx = 0
for list_nodes in [candidate_nodes[n] for n in names]:
    r_indices = []
    for nd in list_nodes:
        flat_nodes.append(nd)
        r_indices.append(global_idx)
        global_idx += 1
    enclosure_to_indices_map.append(r_indices)

N = len(flat_nodes)

# Create the extended distance matrix
dm = np.zeros((N, N), dtype=int)
for i, u in enumerate(flat_nodes):
    for j, v in enumerate(flat_nodes):
        if i != j:
            dm[i][j] = dist_between_nodes.get((u, v), 10**8)

### 3.1. Configuring the OR-Tools Solver

We now configure our `RoutingModel`. Because some enclosures have more than one valid parking candidate within a reasonable radius, we are solving a **Generalized TSP**.
We use the `AddDisjunction` feature of Google OR-Tools to tell the engine that it only needs to pick **exactly one** valid parking slot per enclosure cluster, while minimizing the overall tour distance.

In [32]:
# Configure the OR-Tools model
# Node 0 will be the Main Entrance and we must return to it to complete the TSP cycle
mgr = pywrapcp.RoutingIndexManager(N, 1, 0)
mdl = pywrapcp.RoutingModel(mgr)

def cb(from_idx, to_idx):
    return int(dm[mgr.IndexToNode(from_idx)][mgr.IndexToNode(to_idx)])

cb_idx = mdl.RegisterTransitCallback(cb)
mdl.SetArcCostEvaluatorOfAllVehicles(cb_idx)

# GENERALIZED TSP MAGIC
# For each enclosure starting from the entrance, we allow the engine to choose exactly ONE parking lot
non_visit_penalty = 10**8  # Use a high cost to FORCE a visit

for r_indices in enclosure_to_indices_map[1:]:
    routes_nodes = [mgr.NodeToIndex(i) for i in r_indices]
    mdl.AddDisjunction(routes_nodes, non_visit_penalty)

# Search parameters
params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
params.time_limit.seconds = 60

# A single execution solves everything!
print(f"Evaluating {N} nodes with OR-Tools (Disjunctions) in a single pass...")
sol = mdl.SolveWithParameters(params)

if not sol:
    print("No solution found.")
else:
    best_distance = sol.ObjectiveValue()
    print(f"✅ Best total distance found: {best_distance} meters")
    
    # Extract the optimal route
    idx = mdl.Start(0)
    route_indices = []
    
    while not mdl.IsEnd(idx):
        node_idx = mgr.IndexToNode(idx)
        route_indices.append(node_idx)
        idx = sol.Value(mdl.NextVar(idx))
    # For the last node, which is the end
    route_indices.append(mgr.IndexToNode(idx))
    
    # Associate the chosen flat indices with the enclosure they represent
    best_combo_nodes = [flat_nodes[i] for i in route_indices[:-1]] # excluding the return home
    
    # Build optimal_order and target_nodes for compatibility with Folium rendering code
    optimal_order = []
    for chosen_flat_idx in route_indices[:-1]:
        # Find to which enclosure this chosen_flat_idx belongs
        for pos_enclosure, r_indices in enumerate(enclosure_to_indices_map):
            if chosen_flat_idx in r_indices:
                optimal_order.append(pos_enclosure)
                break
    # Add end
    optimal_order.append(0)

    # Note: target_nodes is ordered in the same way as "names", this is key for the next folium cell.
    # Folium requires "target_nodes" where target_nodes[i] is the parking node chosen for enclosure names[i].
    target_nodes = [None] * len(names)
    for pos_enclosure, chosen_flat_idx in zip(optimal_order[:-1], route_indices[:-1]):
        target_nodes[pos_enclosure] = flat_nodes[chosen_flat_idx]

    print("Visit order (indices):", optimal_order)
    print(" -> ".join(names[i] for i in optimal_order))

Evaluating 53 nodes with OR-Tools (Disjunctions) in a single pass...
✅ Best total distance found: 19838 meters
Visit order (indices): [0, 5, 10, 20, 19, 28, 23, 17, 25, 22, 24, 29, 26, 14, 21, 15, 13, 18, 16, 27, 12, 4, 6, 11, 8, 1, 9, 3, 7, 2, 30, 0]
Main Entrance -> Gaur -> Cebra -> Oso -> Hipopótamos pigmeos -> Guepardo -> Lobo -> Oryx del Cabo -> Wallaby de Bennet y Emú -> León -> Papion de Guinea -> Linces -> Dromedario -> Bisonte Europeo -> Cobos de agua -> Cebras Grevy -> Addax y Búfalo Cafre -> Watusi -> Elefante Africano, Búfalo de agua y Cobo de Lechwe -> Aves rapaces/Granja -> Jirafa, Avestruz y Eland -> Fauna Ibérica -> Gorila/Tigre -> Hipopótamo -> Vaca Tudanca -> Vaca y Caballo Monchino -> Asno Somalí -> Camello Bactriano -> Rinoceronte Blanco -> Llama -> Reptilario/Leones Marinos -> Main Entrance


## 4. Topological Reconstruction and Rendering

The *solver* has returned the logical order (e.g., 0 $\rightarrow$ 3 $\rightarrow$ 1). Now we recover the intermediate road nodes (using `paths_between_nodes`) to draw the exact polygonal route on the map.

In [33]:
# Reconstruct points_of_interest for the best combination: name → [lon, lat] of the chosen parking
points_of_interest = {}
for name, chosen_node in zip(names, target_nodes):
    for pt in candidates_utm[name]:
        if pt_to_node[pt] == chosen_node:
            lon, lat = pt_to_lonlat[pt]
            points_of_interest[name] = [lon, lat]
            break

# Reconstruct the complete route from the pre-computed paths
full_path_nodes = []
for i in range(len(optimal_order) - 1):
    u_idx = optimal_order[i]
    v_idx = optimal_order[i + 1]
    node_u = target_nodes[u_idx]
    node_v = target_nodes[v_idx]
    segment = paths_between_nodes[(node_u, node_v)]
    if i > 0:
        segment = segment[1:]
    full_path_nodes.extend(segment)

# G_metros preserves the original 'lat' and 'lon' in each node
route_coordinates = [[G_metros.nodes[node]['lat'], G_metros.nodes[node]['lon']] for node in full_path_nodes]

### 4.1. Rendering the Interactive Map

Using **Folium**, we will now overlay the `route_coordinates` dynamically onto the map of Cabárceno. We group enclosures sharing the same parking lot to avoid overlapping UI markers, displaying multiple animals underneath a single visit stop.

In [34]:
# Draw the interactive map
mapa = folium.Map(location=[43.350, -3.852], zoom_start=14, tiles='cartodbpositron')

# Route line
folium.PolyLine(route_coordinates, color="#FF5A5F", weight=5, opacity=0.8).add_to(mapa)

# Map each enclosure to its visit number (position in the route, 1-based)
name_to_order = {names[idx]: pos + 1 for pos, idx in enumerate(optimal_order[:-1])}

# Group enclosures by parking coordinates to avoid overlapping markers
coords_to_names = defaultdict(list)
for name, coords in points_of_interest.items():
    coords_to_names[tuple(coords)].append(name)

for coords, group_names in coords_to_names.items():
    is_entrance = "Main Entrance" in group_names
    bg_color = '#2ca02c' if is_entrance else '#1f77b4'

    nums = sorted(name_to_order[n] for n in group_names)
    num_label = "/".join(str(n) for n in nums)

    popup_text = "<br>".join(f"<b>{n}</b>" for n in group_names)

    folium.Marker(
        location=[coords[1], coords[0]],
        popup=folium.Popup(popup_text, max_width=200),
        icon=folium.DivIcon(
            html=f'''<div style="
                background-color: {bg_color};
                color: white;
                border-radius: 50%;
                width: 28px;
                height: 28px;
                display: flex;
                align-items: center;
                justify-content: center;
                font-weight: bold;
                font-size: 12px;
                border: 2px solid white;
                box-shadow: 1px 1px 3px rgba(0,0,0,0.5);
            ">{num_label}</div>''',
            icon_size=(28, 28),
            icon_anchor=(14, 14)
        )
    ).add_to(mapa)

# Save and open the map in the browser
output_path = os.path.abspath('../data/processed/optimal_route.html')
mapa.save(output_path)
webbrowser.open(f'file://{output_path}')
print(f"Map saved at: {output_path}")

Map saved at: /Users/david/Documents/cabarceno-routing-optimization/data/processed/optimal_route.html


In [35]:
import pickle

export_data = {
    "names": names,
    "candidate_nodes": candidate_nodes,
    "dist_between_nodes": dist_between_nodes,
    "paths_between_nodes": paths_between_nodes,
    "pt_to_lonlat": pt_to_lonlat,
    "candidates_utm": candidates_utm,
    "pt_to_node": pt_to_node,
    "optimal_distance": best_distance,
    "optimal_route_names": [names[i] for i in optimal_order]
}

with open('../data/processed/optimization_results.pkl', 'wb') as f:
    pickle.dump(export_data, f)
    
print("Saved optimization_results.pkl with names:", names)

Saved optimization_results.pkl with names: ['Main Entrance', 'Vaca y Caballo Monchino', 'Llama', 'Camello Bactriano', 'Fauna Ibérica', 'Gaur', 'Gorila/Tigre', 'Rinoceronte Blanco', 'Vaca Tudanca', 'Asno Somalí', 'Cebra', 'Hipopótamo', 'Jirafa, Avestruz y Eland', 'Addax y Búfalo Cafre', 'Bisonte Europeo', 'Cebras Grevy', 'Elefante Africano, Búfalo de agua y Cobo de Lechwe', 'Oryx del Cabo', 'Watusi', 'Hipopótamos pigmeos', 'Oso', 'Cobos de agua', 'León', 'Lobo', 'Papion de Guinea', 'Wallaby de Bennet y Emú', 'Dromedario', 'Aves rapaces/Granja', 'Guepardo', 'Linces', 'Reptilario/Leones Marinos']
